In [1]:
# ================================================================
#  LCCN-R16b - TRAIN-ONLY RESOURCE: XLM-RoBERTa, 3 SEEDS  [GPU, ~2.5 h]
#  Trains XLM-RoBERTa on lccn_trainonly at seeds 42, 43, 44 (per-seed
#  prediction files) and compares each seed with the stored lccn
#  predictions of the same seed (paired bootstrap, 10,000 resamples).
#  Add Input: lccn-r16a-trainonly-arms (output), paper1-nb1-lccn-final
#             (output), paper1-nb2-lccn-ml (output),
#             paper1-nb4-lccn-xlmroberta (output, lccn seed 42),
#             paper1-nb6-lccn-xlm43-44 (output, lccn seed 44),
#             lccn-r22c-missing-seeds (output, lccn seed 43).
#  Accelerator GPU. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys, os, shutil, glob
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q scikit-learn xgboost scipy')
sh('pip install -q transformers accelerate --upgrade')
for p in ("trainer.py", "/kaggle/working/trainer.py"):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("__pycache__", ignore_errors=True); sys.modules.pop("trainer", None)
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "trainer.py not found - attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
import torch, transformers
print("\npython", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU is off - Settings > Accelerator > GPU"
print("gpu:", torch.cuda.get_device_name(0))


+ pip install -q scikit-learn xgboost scipy
+ pip install -q transformers accelerate --upgrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 39.5 MB/s eta 0:00:00

python 3.13.15 | torch 2.11.0+cu128 | transformers 5.18.0 | cuda True
gpu: Tesla T4


In [2]:
# ================================================================
#  CELL 1 - SETUP AND ARM JOIN
# ================================================================
import importlib, trainer
importlib.reload(trainer)
from trainer import *
import pandas as pd, numpy as np, re
FP = trainer.fingerprint(); assert trainer.TRAINER_VERSION == "1.4" and FP == "5b607248090bd26e", FP
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
tt = glob.glob("/kaggle/input/**/trainonly_texts.csv", recursive=True); assert tt, "attach lccn-r16a output"
g = pd.read_csv(tt[0], dtype=str, keep_default_na=False); g.columns = [c.strip().lower() for c in g.columns]
idc = next(c for c in g.columns if c in ("id", "comment_id"))
for f in (tr, va, te):
    f["text_lccn_trainonly"] = f["id"].astype(str).map(dict(zip(g[idc].astype(str), g["text_lccn_trainonly"])))
    assert f["text_lccn_trainonly"].notna().all()
print("arm joined: lccn_trainonly")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
arm joined: lccn_trainonly


In [3]:
# ================================================================
#  CELL 2 - THREE SEEDS, PER-SEED PREDICTIONS, PAIRED BOOTSTRAP VS lccn
# ================================================================
from sklearn.metrics import f1_score
MODEL = "XLM-RoBERTa"; ARM = "lccn_trainonly"
have = {os.path.basename(f) for f in glob.glob("/kaggle/input/**/preds/*__seed*.csv", recursive=True)
        + glob.glob("/kaggle/working/preds/*__seed*.csv")}
rows = []
for seed in (42, 43, 44):
    tag_file = f"{MODEL}__{ARM}__seed{seed}.csv"
    if tag_file in have: print("skip", tag_file); continue
    p = f"/kaggle/working/preds/{MODEL}__{ARM}.csv"
    if os.path.exists(p): os.remove(p)
    r = run_grid(tr, va, te, [ARM], [MODEL], tag=f"lccn_r16b_seed{seed}", seeds_tf=(seed,), resume=False)
    shutil.copy(p, f"/kaggle/working/preds/{tag_file}")
    rows.append(dict(arm=ARM, seed=seed, macro_f1=round(float(r.iloc[0][ARM]), 4)))
pd.DataFrame(rows).to_csv("/kaggle/working/results_lccn_r16b_seeds.csv", index=False)
print(pd.DataFrame(rows).to_string(index=False))

def load_pred(p):
    d = pd.read_csv(p, dtype=str, keep_default_na=False); d.columns = [c.strip().lower() for c in d.columns]
    i = next(c for c in d.columns if c in ("id", "comment_id")); y = next(c for c in d.columns if "true" in c or c == "gold")
    pr = next(c for c in d.columns if "pred" in c); d = d[[i, y, pr]].copy(); d.columns = ["id", "y", "p"]
    for c in ("y", "p"): d[c] = d[c].str.strip().str.lower()
    return d.set_index("id")
def lccn_pred(seed):
    for f in glob.glob("/kaggle/input/**/preds/XLM-RoBERTa__lccn*.csv", recursive=True):
        b = os.path.basename(f)
        if b == f"XLM-RoBERTa__lccn__seed{seed}.csv": return f
        if b == "XLM-RoBERTa__lccn.csv" and ((seed == 42 and "nb4" in f) or (seed == 44 and "nb6" in f)): return f
    return None
rng = np.random.default_rng(0); B = 10000
print("\nlccn_trainonly vs lccn, within seed (paired bootstrap):")
for seed in (42, 43, 44):
    lp = lccn_pred(seed); tp = f"/kaggle/working/preds/{MODEL}__{ARM}__seed{seed}.csv"
    cand = [f for f in glob.glob("/kaggle/input/**/preds/*.csv", recursive=True) if os.path.basename(f) == os.path.basename(tp)]
    if not os.path.exists(tp) and cand: tp = cand[0]
    if lp is None or not os.path.exists(tp): print(f"  seed {seed}: missing predictions"); continue
    A, Bd = load_pred(lp), load_pred(tp); j = A.join(Bd, lsuffix="_a", rsuffix="_b", how="inner"); assert len(j) == 6000
    y, pa, pb = j.y_a.values, j.p_a.values, j.p_b.values; n = len(y)
    obs = f1_score(y, pb, average="macro") - f1_score(y, pa, average="macro")
    d = np.array([f1_score(y[i], pb[i], average="macro") - f1_score(y[i], pa[i], average="macro")
                  for i in (rng.integers(0, n, n) for _ in range(B))])
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"  seed {seed}: lccn {f1_score(y, pa, average='macro'):.4f}  trainonly {f1_score(y, pb, average='macro'):.4f}  "
          f"delta {obs:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
print("\nDONE - results_lccn_r16b_seeds.csv and preds/ written; transcribe only from the committed log.")



=== XLM-RoBERTa / lccn_trainonly ===


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.6757
    epoch  2/10  val macro-F1 = 0.7860
    epoch  3/10  val macro-F1 = 0.7890
    epoch  4/10  val macro-F1 = 0.8055
    epoch  5/10  val macro-F1 = 0.8046
    epoch  6/10  val macro-F1 = 0.8150
    epoch  7/10  val macro-F1 = 0.8235
    epoch  8/10  val macro-F1 = 0.8138
    epoch  9/10  val macro-F1 = 0.8170
    epoch 10/10  val macro-F1 = 0.8175
    early stop at epoch 10
  -> macro-F1 0.8060

wrote /kaggle/working/results_lccn_r16b_seed43.csv

=== XLM-RoBERTa / lccn_trainonly ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7473
    epoch  2/10  val macro-F1 = 0.7901
    epoch  3/10  val macro-F1 = 0.7968
    epoch  4/10  val macro-F1 = 0.7944
    epoch  5/10  val macro-F1 = 0.8092
    epoch  6/10  val macro-F1 = 0.8107
    epoch  7/10  val macro-F1 = 0.8188
    epoch  8/10  val macro-F1 = 0.8092
    epoch  9/10  val macro-F1 = 0.8107
    epoch 10/10  val macro-F1 = 0.8121
    early stop at epoch 10
  -> macro-F1 0.8090

wrote /kaggle/working/results_lccn_r16b_seed44.csv
           arm  seed  macro_f1
lccn_trainonly    42    0.7985
lccn_trainonly    43    0.8060
lccn_trainonly    44    0.8090

lccn_trainonly vs lccn, within seed (paired bootstrap):
  seed 42: lccn 0.7991  trainonly 0.7985  delta -0.0006  95% CI [-0.0091, +0.0076]
  seed 43: lccn 0.8069  trainonly 0.8060  delta -0.0009  95% CI [-0.0077, +0.0061]
  seed 44: lccn 0.8066  trainonly 0.8090  delta +0.0024  95% CI [-0.0048, +0.0097]

DONE - results_lccn_r16b_seeds.csv and preds/ written; transcribe only from the